In [71]:
# Represent (partial) triangulations with sets of edges

# Shortcut for writing edges
def e(x,y):
    return frozenset({x,y})

# Class for (partial or complete) triangulations of
# polygons
class PolyTri:
    # Constructor for an n-gon with optional additional edges
    def __init__(self, n, edges=frozenset()):
        self.n = n
        self.edges = frozenset(e(i%n+1,(i+1)%n+1) for i in range(n))
        self.edges |= edges

    def __eq__(self, other):
        return self.edges == other.edges

    def __hash__(self):
        return hash(self.edges)

    def is_external(self, e):
        assert(len(e) == 2)
        n = self.n
        e = tuple(e)
        d = (e[0] - e[1]) % n
        return (d == 1 or d == n-1)

    def internal_edges(self):
        return frozenset({e for e in self.edges if not self.is_external(e)})

    def get_quad(self, e):
        assert(not self.is_external(e))
        incident_edges = [ee for ee in self.edges - {e} if e & ee]
        q = frozenset({v for e1 in incident_edges for e2 in incident_edges for v in e1 & e2 if e1 != e2})
        assert(len(q) == 4)
        return q

        # Edge flip mutation of triangulation at a given edge
    def mutate(self, e):
        q = self.get_quad(e)
        ee = q - e
        return PolyTri(self.n, (self.edges - {e}) | {ee})

    def adjacent_edges(self, e):
        q = self.get_quad(e)
        return [ee for ee in self.edges - {e} if not self.is_external(ee) and ee < q]

    def __repr__(self):
        return "{}-gon w/ edges ".format(self.n) + "{" + ", ".join(str(tuple(e)) for e in self.internal_edges()) + "}"

# Initial partial triangulation
init_pent = PolyTri(5, {e(1,4), e(1,3)})

In [72]:
init_pent.mutate(e(1,4))

5-gon w/ edges {(1, 3), (3, 5)}

In [73]:
hept_example = PolyTri(7, {e(1,3), e(3,5), e(3,6), e(1,6)})
hept_example.mutate(e(3,6))

7-gon w/ edges {(1, 6), (1, 3), (1, 5), (3, 5)}

In [74]:
hept_example.adjacent_edges(e(3,6))

[frozenset({3, 5}), frozenset({1, 6}), frozenset({1, 3})]

In [75]:
# Class for resolutions of partial triangulations
class Resolution:
    def __init__(self, tri, edge):
        self.tri = PolyTri(tri.n, tri.edges - {edge})
        self.edge = edge

    def __repr__(self):
        return repr(self.tri) + " resolved at " + str(tuple(self.edge))

    def __eq__(self, other):
        return self.tri == other.tri and self.edge == other.edge

    def __hash__(self):
        return hash((self.tri.edges, self.edge))

    def resolve(self):
        return PolyTri(self.tri.n, self.tri.edges | {self.edge})

    def adjacent_resolutions(self):
        e = self.edge
        t = self.resolve()
        cols = set()
        for ee in t.adjacent_edges(e):
            tt = t.mutate(ee)
            assert(len(tt.edges-t.edges) == 1)
            cols.add(Resolution(tt, e))
        return cols

    def relaxation_class(self):
        visited = {self}
        remaining = self.adjacent_resolutions()
        while remaining:
            rr = remaining.pop()
            visited.add(rr)
            remaining |= rr.adjacent_resolutions() - visited
        return visited

    def mutate(self):
        t = self.resolve()
        e = self.edge
        tt = t.mutate(e)
        assert(len(tt.edges-t.edges) == 1)
        ee = list(tt.edges - t.edges)[0]
        assert(tt.edges-{ee} == self.tri.edges)
        return Resolution(self.tri, ee)

Resolution(hept_example2, e(3,6)).mutate()

7-gon w/ edges {(1, 6), (1, 3), (3, 5)} resolved at (1, 5)

In [76]:
def mutate_resolution(r):
    t = resolve(r)
    e = r[1]
    tt = mutate(t, e)
    assert(len(tt-t) == 1)
    ee = list(tt - t)[0]
    assert(tt-{ee} == r[0])
    return (r[0], ee)

# Pretty print resolution
def ppr(r):
    return "({}, {})".format(pp(r[0]), set(r[1]))

def propagate_resolutions(r, rz):
    for rr in r.relaxation_class() - {r}:
        mutated = rr.mutate()
        print("{} requires {}".format(r, mutated))
        if rr.tri in rz:
            if rz[rr.tri] != mutated:
                print("Conflicting resolution for {}!".format(rr.tri))
                raise Exception()
        else:
            rz[rr.tri] = mutated
            propagate_resolutions(mutated, rz)

# Wrapper to avoid constructing initial resolution map
def find_resolutions(r):
    rz = {}
    rz[r.tri] = r
    propagate_resolutions(r, rz)
    return rz

In [77]:
find_resolutions(Resolution(hept_example2, e(3,6)))

7-gon w/ edges {(1, 6), (1, 3), (3, 5)} resolved at (3, 6) requires 7-gon w/ edges {(1, 6), (4, 6), (2, 6)} resolved at (2, 4)
7-gon w/ edges {(1, 6), (4, 6), (2, 6)} resolved at (2, 4) requires 7-gon w/ edges {(1, 6), (1, 4), (4, 6)} resolved at (1, 3)
7-gon w/ edges {(1, 6), (1, 4), (4, 6)} resolved at (1, 3) requires 7-gon w/ edges {(1, 6), (3, 6), (4, 6)} resolved at (2, 6)
7-gon w/ edges {(1, 6), (3, 6), (4, 6)} resolved at (2, 6) requires 7-gon w/ edges {(2, 4), (2, 7), (4, 6)} resolved at (4, 7)
7-gon w/ edges {(2, 4), (2, 7), (4, 6)} resolved at (4, 7) requires 7-gon w/ edges {(2, 7), (5, 7), (3, 7)} resolved at (3, 5)
7-gon w/ edges {(2, 7), (5, 7), (3, 7)} resolved at (3, 5) requires 7-gon w/ edges {(3, 6), (2, 7), (3, 7)} resolved at (4, 6)
7-gon w/ edges {(3, 6), (2, 7), (3, 7)} resolved at (4, 6) requires 7-gon w/ edges {(2, 7), (3, 7), (4, 7)} resolved at (5, 7)
7-gon w/ edges {(2, 7), (3, 7), (4, 7)} resolved at (5, 7) requires 7-gon w/ edges {(2, 7), (2, 5), (3, 5)} res

Exception: 

In [79]:
find_resolutions(Resolution(init_pent2, e(1,3)))

5-gon w/ edges {(1, 4)} resolved at (1, 3) requires 5-gon w/ edges {(3, 5)} resolved at (2, 5)
5-gon w/ edges {(3, 5)} resolved at (2, 5) requires 5-gon w/ edges {(2, 4)} resolved at (1, 4)
5-gon w/ edges {(2, 4)} resolved at (1, 4) requires 5-gon w/ edges {(1, 3)} resolved at (3, 5)
5-gon w/ edges {(1, 3)} resolved at (3, 5) requires 5-gon w/ edges {(2, 5)} resolved at (2, 4)
5-gon w/ edges {(2, 5)} resolved at (2, 4) requires 5-gon w/ edges {(1, 4)} resolved at (1, 3)


{5-gon w/ edges {(1, 4)}: 5-gon w/ edges {(1, 4)} resolved at (1, 3),
 5-gon w/ edges {(3, 5)}: 5-gon w/ edges {(3, 5)} resolved at (2, 5),
 5-gon w/ edges {(2, 4)}: 5-gon w/ edges {(2, 4)} resolved at (1, 4),
 5-gon w/ edges {(1, 3)}: 5-gon w/ edges {(1, 3)} resolved at (3, 5),
 5-gon w/ edges {(2, 5)}: 5-gon w/ edges {(2, 5)} resolved at (2, 4)}

In [78]:

find_resolutions(Resolution(PolyTri(6, {e(1,5),e(1,3)}), e(3,5)))

6-gon w/ edges {(1, 5), (1, 3)} resolved at (3, 5) requires 6-gon w/ edges {(1, 5), (2, 5)} resolved at (2, 4)
6-gon w/ edges {(1, 5), (2, 5)} resolved at (2, 4) requires 6-gon w/ edges {(1, 4), (1, 5)} resolved at (1, 3)
6-gon w/ edges {(1, 4), (1, 5)} resolved at (1, 3) requires 6-gon w/ edges {(1, 5), (3, 5)} resolved at (2, 5)
6-gon w/ edges {(1, 5), (3, 5)} resolved at (2, 5) requires 6-gon w/ edges {(2, 4), (1, 5)} resolved at (1, 4)
6-gon w/ edges {(2, 4), (1, 5)} resolved at (1, 4) requires 6-gon w/ edges {(4, 6), (1, 3)} resolved at (3, 6)
6-gon w/ edges {(4, 6), (1, 3)} resolved at (3, 6) requires 6-gon w/ edges {(1, 3), (3, 5)} resolved at (1, 5)
6-gon w/ edges {(1, 3), (3, 5)} resolved at (1, 5) requires 6-gon w/ edges {(1, 4), (1, 3)} resolved at (4, 6)
6-gon w/ edges {(1, 4), (1, 3)} resolved at (4, 6) requires 6-gon w/ edges {(3, 6), (1, 3)} resolved at (3, 5)
6-gon w/ edges {(3, 6), (1, 3)} resolved at (3, 5) requires 6-gon w/ edges {(1, 5), (2, 5)} resolved at (2, 4)
6

Exception: 

In [86]:
# Generate all triangulations of an n-gon
def generate_triangulations(n):
    # initial triangulation
    t = PolyTri(n, {e(1,j+3) for j in range(0, n-3)})
    visited = {t}
    remaining = {t.mutate(e) for e in t.internal_edges()}
    while remaining:
        tt = remaining.pop()
        visited |= {tt}
        remaining |= {tt.mutate(e) for e in tt.internal_edges()} - visited
    return visited

def generate_resolutions(n):
    return {Resolution(t, e) for t in generate_triangulations(n) for e in t.internal_edges()}

In [85]:
len(generate_triangulations(7))

42

In [108]:
def exchange_statistics(n):
    rs = generate_resolutions(n)
    x = len(rs)
    y = len({frozenset(r.relaxation_class()) for r in rs})
    return (n, int(x/(n-3)), int(x/2), y)

[exchange_statistics(n) for n in range(4,12)]

[(4, 2, 1, 2),
 (5, 5, 5, 5),
 (6, 14, 21, 15),
 (7, 42, 84, 49),
 (8, 132, 330, 168),
 (9, 429, 1287, 594),
 (10, 1430, 5005, 2145),
 (11, 4862, 19448, 7865)]